# Multi-modal bone marrow integration with RegularizedMultimodalVI (early stopping)

This notebook uses `RegularizedMultimodalVI` with the same architecture as the main multimodal tutorial,
but with **early stopping** and **checkpoint saving** every 1000 epochs.

Training uses `train_size=0.9` (90/10 train/val split), `early_stopping_patience=20`,
and `max_epochs=5000` as upper bound.

Compare with:
- [bone_marrow_multimodal_tutorial.ipynb](bone_marrow_multimodal_tutorial.ipynb) — same model trained for 3500 epochs (no early stopping)


In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import torch
import mudata as mu

import scvi
import regularizedvi

# Simple text progress bar (works with papermill)
scvi.settings.progress_bar_style = "tqdm"

# Use high-precision matmul for better numerical stability on GPU
torch.set_float32_matmul_precision("high")

rcParams["pdf.fonttype"] = 42  # enables correct plotting of text for publication figures

## 1. Load data

Load the bone marrow multiome dataset. This dataset contains both GEX (gene expression)
and ATAC (chromatin accessibility) features in a single AnnData, distinguished by the
`feature_types` column in `.var`.

In [ ]:
import os

from regularizedvi.utils import download_bone_marrow_dataset

results_folder = "results/multimodal_tutorial_early_stopping/"
os.makedirs(results_folder, exist_ok=True)

h5ad_path = download_bone_marrow_dataset(data_folder="data/")
adata_full = sc.read_h5ad(h5ad_path)

print(adata_full)
print(f"\nFeature types: {adata_full.var['feature_types'].value_counts().to_dict()}")
print(f"Batches: {adata_full.obs['batch'].nunique()}")
print(f"Sites: {adata_full.obs['site'].nunique()}")
print(f"Donors: {adata_full.obs['donor'].nunique()}")

## 2. Quality control

Apply the same cell QC as the RNA-only tutorial: filter on UMI counts, genes detected,
ATAC fragments, mitochondrial fraction, and doublet score.

In [ ]:
# Mitochondrial fraction (computed on GEX features)
gex_mask = adata_full.var["feature_types"] == "GEX"
adata_full.var["SYMBOL"] = adata_full.var_names.values.copy()

gex_adata = adata_full[:, gex_mask].copy()
gex_adata.var["mt"] = [gene.startswith("MT-") for gene in gex_adata.var["SYMBOL"]]
mt_counts = gex_adata[:, gex_adata.var["mt"].tolist()].X.sum(1).A.squeeze()
adata_full.obs["mt_frac"] = mt_counts / adata_full.obs["GEX_n_counts"]

del gex_adata

In [ ]:
# Doublet detection (on GEX)
from regularizedvi.utils import filter_genes

adata_gex_for_scrub = adata_full[:, gex_mask].copy()
adata_gex_for_scrub.var_names = adata_gex_for_scrub.var["gene_ids"].values.astype(str).copy()
adata_gex_for_scrub.X = adata_gex_for_scrub.layers["counts"]

selected = filter_genes(
    adata_gex_for_scrub,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.05,
    nonz_mean_cutoff=1.07,
)
adata_for_scrublet = adata_gex_for_scrub[:, selected].copy()
sc.pp.scrublet(
    adata_for_scrublet,
    batch_key="batch",
    n_prin_comps=40,
    verbose=False,
)
adata_full.obs["doublet_score"] = adata_for_scrublet.obs["doublet_score"]
del adata_gex_for_scrub, adata_for_scrublet

print(f"Fraction above 0.18 threshold: {(adata_full.obs['doublet_score'] > 0.18).mean():.3f}")

In [ ]:
# Cell filtering
n_before = adata_full.n_obs
cell_mask = (
    (adata_full.obs["GEX_n_genes"] > 500)
    & (adata_full.obs["GEX_n_counts"] > 1000)
    & (adata_full.obs["GEX_n_counts"] < 80000)
    & (adata_full.obs["GEX_n_genes"] < 10000)
    & (adata_full.obs["ATAC_atac_fragments"] > 1000)
    & (adata_full.obs["ATAC_atac_fragments"] < 100000)
    & (adata_full.obs["mt_frac"] < 0.20)
    & (adata_full.obs["doublet_score"] < 0.18)
)
adata_full = adata_full[cell_mask, :].copy()
print(f"Filtered {n_before} -> {adata_full.n_obs} cells ({n_before - adata_full.n_obs} removed)")

## 3. Split into RNA and ATAC modalities

Split the combined AnnData into separate modalities for MuData creation.

In [ ]:
# Split by feature type
gex_mask = adata_full.var["feature_types"] == "GEX"
atac_mask = adata_full.var["feature_types"] == "ATAC"

adata_rna = adata_full[:, gex_mask].copy()
adata_atac = adata_full[:, atac_mask].copy()

# Use counts as X
adata_rna.X = adata_rna.layers["counts"]
adata_atac.X = adata_atac.layers["counts"]

# Use gene symbols for RNA var_names
adata_rna.var["SYMBOL"] = adata_rna.var_names.values.copy()
adata_rna.var_names = adata_rna.var["gene_ids"].values.astype(str).copy()

print(f"RNA: {adata_rna.shape}")
print(f"ATAC: {adata_atac.shape}")

## 4. Feature selection

### RNA gene selection

Select informative genes using `filter_genes` (adapted from cell2location).

In [ ]:
selected_genes = filter_genes(
    adata_rna,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.03,
)
adata_rna = adata_rna[:, selected_genes].copy()
print(f"Selected {adata_rna.n_vars} RNA genes")

### ATAC peak selection

We use the same `filter_genes` tool for ATAC peaks. The ATAC data contains fragment
counts per peak per cell (not binary) — the most common non-zero value is 2 (both ends
of a sequencing fragment), with a typical non-zero mean of ~2–3 fragments.

The cutoffs operate the same way as for RNA:
- `cell_count_cutoff` — peaks accessible in at least this many cells (the lower tier)
- `cell_percentage_cutoff2` — peaks in at least this fraction of cells are always kept
- `nonz_mean_cutoff` — mean fragment count in accessible cells (filters low-signal peaks)

For ATAC peaks, `log10(nonz_mean) ≈ 0.3` (i.e. mean ≈ 2 fragments), so a lower
`nonz_mean_cutoff` than for RNA is appropriate.

In [ ]:
selected_peaks = filter_genes(
    adata_atac,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.01,
    nonz_mean_cutoff=1.03,  # mean fragment count > 1.03 in accessible cells
)
adata_atac = adata_atac[:, selected_peaks].copy()
print(f"Selected {adata_atac.n_vars} ATAC peaks")

## 5. Create MuData

Combine the RNA and ATAC AnnData objects into a MuData for multi-modal modelling.

In [ ]:
mdata = mu.MuData({"rna": adata_rna, "atac": adata_atac})
print(mdata)
print(f"\nRNA: {mdata.mod['rna'].shape}")
print(f"ATAC: {mdata.mod['atac'].shape}")

## 6. Model setup

### The modality dictionary interface

`RegularizedMultimodalVI` uses a **dictionary convention** for per-modality configuration:

```python
# Scalar values apply to ALL modalities:
n_hidden=256  # both RNA and ATAC encoders/decoders have 256 hidden units

# Dict values are per-modality:
n_hidden={'rna': 512, 'atac': 256}  # RNA gets 512, ATAC gets 256
n_latent={'rna': 64, 'atac': 16}    # RNA encoder outputs 64-dim Z, ATAC outputs 16-dim Z
```

With `latent_mode='concatenation'` (default), the total latent dimension is the sum:
Z = [Z_rna (64); Z_atac (16)] = 80 dimensions. All decoders see the full 80-dim Z.

### Per-modality flags

Two lists control which modalities get special processing:

- `additive_background_modalities=['rna']` — ambient RNA correction for RNA only
  (ATAC has negligible ambient signal: ~0.1% of reads in empty droplets)
- `region_factors_modalities=['atac']` — per-peak bias factors for ATAC only
  (corrects for GC content, mappability, peak width)

In [ ]:
regularizedvi.RegularizedMultimodalVI.setup_mudata(
    mdata,
    batch_key="batch",
    categorical_covariate_keys=["site", "donor"],
)

In [ ]:
model = regularizedvi.RegularizedMultimodalVI(
    mdata,
    # Per-modality architecture sizes
    n_hidden={"rna": 512, "atac": 256},
    n_latent={"rna": 128, "atac": 64},
    n_layers=1,
    # Z combination: concatenation preserves modality-specific signal
    latent_mode="concatenation",
    # Likelihood: GammaPoisson for both modalities (count-based ATAC)
    likelihood_distribution="gamma_poisson",
    # Per-modality flags (defaults shown explicitly)
    additive_background_modalities=["rna"],  # ambient correction for RNA only
    region_factors_modalities=["atac"],  # per-peak bias for ATAC only
    # Dispersion
    dispersion="gene-batch",
    regularise_dispersion=True,
)

print(model)
print(f"\nTotal latent dim: {model.module.total_latent_dim}")
print(f"  RNA Z dim: {model.module.n_latent_dict['rna']}")
print(f"  ATAC Z dim: {model.module.n_latent_dict['atac']}")

## 7. Train (early stopping)

Training uses `train_size=0.9` (90/10 train/val split) with early stopping
monitoring `elbo_validation` (patience=20). Checkpoints are saved every 1000 epochs.
The `max_epochs=5000` serves as an upper bound.


In [ ]:
from scvi.train import SaveCheckpoint

checkpoint_cb = SaveCheckpoint(
    dirpath=f"{results_folder}/checkpoints",
    every_n_epochs=1000,
    save_top_k=-1,
    filename="epoch-{epoch}",
)

model.train(
    check_val_every_n_epoch=1,
    train_size=0.9,
    max_epochs=5000,
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_monitor="elbo_validation",
    enable_checkpointing=True,
    callbacks=[checkpoint_cb],
)

In [ ]:
# Training loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(model.history_["elbo_train"][80:])
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("ELBO")
axes[0].set_title("ELBO (training)")

axes[1].plot(model.history_["reconstruction_loss_train"][80:])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Reconstruction loss")
axes[1].set_title("Reconstruction loss (training)")

plt.tight_layout()
plt.show()

### Per-modality training diagnostics

The model tracks three metrics per modality via `extra_metrics` in the loss function:

- **`recon_loss_{modality}`** — per-modality reconstruction loss (how well each decoder reconstructs its input)
- **`kl_z_{modality}`** — per-modality KL divergence on Z (how much information each encoder pushes into its Z dims)
- **`z_var_{modality}`** — per-modality Z variance (latent utilisation — are all Z dims being used?)

These are automatically logged by scvi-tools and accessible in `model.history_`.

**Diagnostic interpretation**:
- Fast ATAC recon_loss drop + low z_var_atac = **free-riding** (ATAC decoder uses Z_rna dims for cell-type info)
- Slow ATAC recon_loss + low z_var_atac = ATAC genuinely hard to learn
- kl_z per modality = how much info each encoder pushes into its Z dims

In [ ]:
skip_epochs = 80  # skip initial transient

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Per-modality reconstruction loss
ax = axes[0]
for name in model.module.modality_names:
    key = f"recon_loss_{name}_train"
    if key in model.history_:
        ax.plot(model.history_[key].iloc[skip_epochs:].values, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Reconstruction loss")
ax.set_title("Per-modality reconstruction loss")
ax.legend()

# Per-modality KL on Z
ax = axes[1]
for name in model.module.modality_names:
    key = f"kl_z_{name}_train"
    if key in model.history_:
        ax.plot(model.history_[key].iloc[skip_epochs:].values, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("KL divergence on Z")
ax.set_title("Per-modality KL(q(z) || p(z))")
ax.legend()

# Per-modality Z variance (latent utilisation)
ax = axes[2]
for name in model.module.modality_names:
    key = f"z_var_{name}_train"
    if key in model.history_:
        ax.plot(model.history_[key].iloc[skip_epochs:].values, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean Z variance")
ax.set_title("Per-modality latent utilisation (Z variance)")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save model
ref_run_name = f"{results_folder}/model"
model.save(ref_run_name, overwrite=True)

## 8. Three paths to UMAP

With `latent_mode='concatenation'`, Z = [Z_rna; Z_atac]. We extract the full latent
representation and split it into modality-specific slices for separate visualisations.

This reveals which aspects of cell identity are captured by each modality:
- **Joint Z** — the complete picture, integrating both modalities
- **RNA Z** — transcriptomic structure (gene expression programs)
- **ATAC Z** — epigenomic structure (chromatin accessibility programs)

Cells that cluster together in RNA Z but separate in ATAC Z (or vice versa) indicate
modality-specific biology — the key advantage of the concatenation strategy over
weighted mean approaches.

In [ ]:
# Extract full latent representation
latent = model.get_latent_representation()
print(f"Full latent shape: {latent.shape}")

# Split into modality-specific components
n_latent_rna = model.module.n_latent_dict["rna"]
n_latent_atac = model.module.n_latent_dict["atac"]

# The order matches model.module.modality_names (sorted alphabetically)
print(f"Modality order in Z: {model.module.modality_names}")

# Split the latent space by modality
# modality_names is sorted, so we need to find the right offsets
z_parts = {}
offset = 0
for name in model.module.modality_names:
    n_lat = model.module.n_latent_dict[name]
    z_parts[name] = latent[:, offset : offset + n_lat]
    offset += n_lat
    print(f"  Z_{name}: dims [{offset - n_lat}, {offset}) -> shape {z_parts[name].shape}")

# Store in the first modality's obs for plotting
adata_rna.obsm["X_multiVI_joint"] = latent
adata_rna.obsm["X_multiVI_rna"] = z_parts["rna"]
adata_rna.obsm["X_multiVI_atac"] = z_parts["atac"]

In [ ]:
k = 50

# --- Joint UMAP ---
sc.pp.neighbors(adata_rna, use_rep="X_multiVI_joint", n_neighbors=k, metric="euclidean", key_added="joint")
sc.tl.umap(adata_rna, min_dist=0.4, spread=1.3, neighbors_key="joint")
adata_rna.obsm["X_umap_joint"] = adata_rna.obsm["X_umap"].copy()

# --- RNA-only UMAP ---
sc.pp.neighbors(adata_rna, use_rep="X_multiVI_rna", n_neighbors=k, metric="euclidean", key_added="rna")
sc.tl.umap(adata_rna, min_dist=0.4, spread=1.3, neighbors_key="rna")
adata_rna.obsm["X_umap_rna"] = adata_rna.obsm["X_umap"].copy()

# --- ATAC-only UMAP ---
sc.pp.neighbors(adata_rna, use_rep="X_multiVI_atac", n_neighbors=k, metric="euclidean", key_added="atac")
sc.tl.umap(adata_rna, min_dist=0.4, spread=1.3, neighbors_key="atac")
adata_rna.obsm["X_umap_atac"] = adata_rna.obsm["X_umap"].copy()

print("Computed 3 UMAP embeddings: joint, RNA-only, ATAC-only")

### Joint Z UMAP

The full Z = [Z_rna; Z_atac] captures both transcriptomic and epigenomic structure.

In [ ]:
color = ["site", "batch", "l1_cell_type", "l2_cell_type"]

rcParams["figure.figsize"] = 7, 7
adata_rna.obsm["X_umap"] = adata_rna.obsm["X_umap_joint"]

sc.pl.umap(
    adata_rna,
    color=color,
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=3,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=10,
    title=[f"Joint Z - {c}" for c in color],
)

### RNA-only Z UMAP

Z_rna captures the transcriptomic component of the latent space.
This should resemble a standard scVI/regularizedVI embedding.

In [ ]:
adata_rna.obsm["X_umap"] = adata_rna.obsm["X_umap_rna"]

sc.pl.umap(
    adata_rna,
    color=color,
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=3,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=10,
    title=[f"RNA Z - {c}" for c in color],
)

### ATAC-only Z UMAP

Z_atac captures the epigenomic component. Differences from the RNA UMAP
indicate chromatin-specific biology: cells with similar transcriptomes
but distinct chromatin landscapes will appear separated here.

In [ ]:
adata_rna.obsm["X_umap"] = adata_rna.obsm["X_umap_atac"]

sc.pl.umap(
    adata_rna,
    color=color,
    color_map="RdPu",
    ncols=1,
    palette=sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy,
    size=3,
    vmin=0,
    vmax="p99.9",
    use_raw=False,
    legend_fontsize=10,
    title=[f"ATAC Z - {c}" for c in color],
)

### Side-by-side comparison

Compare all three embeddings coloured by cell type to see which modality
drives which aspects of cell identity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

palette = dict(
    zip(
        adata_rna.obs["l2_cell_type"].cat.categories,
        (sc.pl.palettes.default_102 + sc.pl.palettes.zeileis_28 + sc.pl.palettes.vega_20_scanpy)[
            : adata_rna.obs["l2_cell_type"].cat.categories.size
        ],
        strict=False,
    )
)

for ax, (umap_key, title) in zip(
    axes,
    [
        ("X_umap_joint", "Joint Z"),
        ("X_umap_rna", "RNA Z"),
        ("X_umap_atac", "ATAC Z"),
    ],
    strict=False,
):
    adata_rna.obsm["X_umap"] = adata_rna.obsm[umap_key]
    sc.pl.umap(
        adata_rna,
        color="l2_cell_type",
        palette=palette,
        size=2,
        ax=ax,
        show=False,
        title=title,
        legend_loc="none",
    )

plt.tight_layout()
plt.show()

## 8b. Decoder attribution analysis (Jacobian-based)

Which latent dimensions does each decoder actually use? We compute the **mean absolute Jacobian**
`|d(px_rate)/d(z)|` per latent dimension for each modality's decoder via finite differences.
This reveals which parts of the shared Z = [Z_rna; Z_atac] inform each modality.

The method returns two things per modality:
- **`attribution`** (n_cells, n_latent) — raw decoder sensitivity per latent dim. Can be analysed directly.
- **`weighted_z`** (n_cells, n_latent) — `z * attribution`, a modality-specific view of the latent space.

**What to look for**:
- If the RNA decoder uses primarily Z_rna dims (first 128) and ATAC decoder uses primarily Z_atac dims (last 64),
  the modalities learned independent representations.
- If the ATAC decoder relies heavily on Z_rna dims, this indicates **free-riding**: the ATAC decoder uses
  RNA-derived cell-type information rather than learning from chromatin data.
- Differences in **attribution UMAPs** between modalities highlight which cell populations are driven by
  RNA vs ATAC biology.

In [ ]:
# Compute decoder attribution per modality
# This runs n_latent forward passes per modality per batch (finite differences)
attribution = model.get_modality_attribution(batch_size=256)

for name in model.module.modality_names:
    attr = attribution[name]["attribution"]
    print(f"{name} attribution shape: {attr.shape}")
    print(f"  Mean importance per Z dim: min={attr.mean(0).min():.4f}, max={attr.mean(0).max():.4f}")

In [ ]:
# Attribution summary: which Z dimensions does each decoder use?
n_latent_rna = model.module.n_latent_dict["rna"]
n_latent_atac = model.module.n_latent_dict["atac"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name in zip(axes, model.module.modality_names, strict=False):
    attr = attribution[name]["attribution"]
    mean_attr = attr.mean(axis=0)  # average over cells

    # Color bars by which modality's Z dims they belong to
    colors = ["tab:blue"] * n_latent_rna + ["tab:orange"] * n_latent_atac
    # Sort alphabetically: atac dims come first, then rna
    offset = 0
    colors = []
    for mod_name in model.module.modality_names:
        n = model.module.n_latent_dict[mod_name]
        c = "tab:blue" if mod_name == "rna" else "tab:orange"
        colors.extend([c] * n)
        offset += n

    ax.bar(range(len(mean_attr)), mean_attr, color=colors, alpha=0.7)
    ax.set_xlabel("Latent dimension")
    ax.set_ylabel("Mean |Jacobian|")
    ax.set_title(f"{name.upper()} decoder attribution")

    # Add modality boundary
    first_mod_size = model.module.n_latent_dict[model.module.modality_names[0]]
    ax.axvline(first_mod_size - 0.5, color="red", linestyle="--", alpha=0.5)

    # Legend
    from matplotlib.patches import Patch

    legend_elements = []
    for mod_name in model.module.modality_names:
        c = "tab:blue" if mod_name == "rna" else "tab:orange"
        legend_elements.append(Patch(facecolor=c, alpha=0.7, label=f"Z_{mod_name}"))
    ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

# Fraction of total attribution from own vs cross-modality Z dims
for name in model.module.modality_names:
    attr = attribution[name]["attribution"]
    total = attr.mean(axis=0).sum()
    offset = 0
    for mod_name in model.module.modality_names:
        n = model.module.n_latent_dict[mod_name]
        frac = attr.mean(axis=0)[offset : offset + n].sum() / total
        print(f"  {name.upper()} decoder: {frac:.1%} of attribution from Z_{mod_name}")
        offset += n
    print()

### Attribution-weighted UMAPs

The **weighted Z** (`z * attribution`) emphasises latent dimensions that each decoder actually uses.
UMAP on weighted Z provides a view of cell heterogeneity *as seen by each decoder*.

Differences between RNA and ATAC attribution UMAPs reveal cells that are driven by
different modalities — the key biological question for multiome data.

In [ ]:
# Attribution-weighted UMAPs
for name in model.module.modality_names:
    adata_rna.obsm[f"X_multiVI_attr_{name}"] = attribution[name]["weighted_z"]

# Compute UMAPs on attribution-weighted Z
for name in model.module.modality_names:
    sc.pp.neighbors(
        adata_rna,
        use_rep=f"X_multiVI_attr_{name}",
        n_neighbors=k,
        metric="euclidean",
        key_added=f"attr_{name}",
    )
    sc.tl.umap(adata_rna, min_dist=0.4, spread=1.3, neighbors_key=f"attr_{name}")
    adata_rna.obsm[f"X_umap_attr_{name}"] = adata_rna.obsm["X_umap"].copy()

print("Computed attribution-weighted UMAP embeddings")

In [ ]:
# Compare: joint Z, RNA attribution-weighted Z, ATAC attribution-weighted Z
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

titles_keys = [
    ("X_umap_joint", "Joint Z"),
    ("X_umap_attr_rna", "RNA decoder attribution Z"),
    ("X_umap_attr_atac", "ATAC decoder attribution Z"),
]

for ax, (umap_key, title) in zip(axes, titles_keys, strict=False):
    adata_rna.obsm["X_umap"] = adata_rna.obsm[umap_key]
    sc.pl.umap(
        adata_rna,
        color="l2_cell_type",
        palette=palette,
        size=2,
        ax=ax,
        show=False,
        title=title,
        legend_loc="none",
    )

plt.tight_layout()
plt.show()

## 9. Marker gene visualisation

Overlay marker genes on the **Joint Z** UMAP to verify that the multi-modal latent
space captures expected biology.

In [ ]:
# Use joint UMAP for marker visualisation
adata_rna.obsm["X_umap"] = adata_rna.obsm["X_umap_joint"]

# Load marker genes
marker_df = pd.read_csv("known_marker_genes.csv")
categories = marker_df.groupby("category", sort=False)["gene"].apply(lambda x: list(dict.fromkeys(x)))

rcParams["figure.figsize"] = 6, 6

for category, gene_list in categories.items():
    present = [g for g in gene_list if g in adata_rna.var["SYMBOL"].values]
    if not present:
        continue

    gene_idx = adata_rna.var["SYMBOL"].isin(present)
    selected_expr = adata_rna[:, gene_idx].X.multiply(1.0 / adata_rna.obs[["GEX_n_counts"]].values)
    selected_expr = selected_expr.toarray() * 1e4

    col_names = [f"{m} normalised" for m in present]
    adata_rna.obs[col_names] = selected_expr

    print(f"\n{'=' * 60}")
    print(f"{category} ({len(present)} genes)")
    print(f"{'=' * 60}")

    sc.pl.umap(
        adata_rna,
        color=col_names,
        color_map="RdPu",
        ncols=5,
        size=3,
        vmin=0,
        vmax="p99.99",
        use_raw=False,
        legend_fontsize=8,
        title=[f"{m}" for m in present],
    )

## 10. Save outputs

In [ ]:
output_dir = f"{ref_run_name}/outputs/"
os.makedirs(output_dir, exist_ok=True)

# Save latent representations
for key, name in [("X_multiVI_joint", "joint"), ("X_multiVI_rna", "rna"), ("X_multiVI_atac", "atac")]:
    df = pd.DataFrame(
        adata_rna.obsm[key],
        index=adata_rna.obs_names,
        columns=range(adata_rna.obsm[key].shape[1]),
    )
    df.to_csv(f"{output_dir}/X_multiVI_{name}.csv")

# Save UMAP coordinates
umap_keys = [
    ("X_umap_joint", "joint"),
    ("X_umap_rna", "rna"),
    ("X_umap_atac", "atac"),
    ("X_umap_attr_rna", "attr_rna"),
    ("X_umap_attr_atac", "attr_atac"),
]
for umap_key, name in umap_keys:
    if umap_key in adata_rna.obsm:
        df = pd.DataFrame(
            adata_rna.obsm[umap_key],
            index=adata_rna.obs_names,
            columns=range(2),
        )
        df.to_csv(f"{output_dir}/X_umap_{name}_k{k}.csv")

# Save attribution data
for name in model.module.modality_names:
    if name in attribution:
        for key in ["attribution", "weighted_z"]:
            df = pd.DataFrame(
                attribution[name][key],
                index=adata_rna.obs_names,
                columns=range(attribution[name][key].shape[1]),
            )
            df.to_csv(f"{output_dir}/attribution_{name}_{key}.csv")

print(f"Outputs saved to {output_dir}")

## Summary

This notebook demonstrated **RegularizedMultimodalVI** with early stopping for joint RNA + ATAC integration:

1. **Data preparation** — split multiome AnnData into separate RNA and ATAC modalities, select informative features, create MuData
2. **Model configuration** — per-modality architecture sizes via dict interface (`n_hidden`, `n_latent`), per-modality flags (`additive_background_modalities`, `region_factors_modalities`)
3. **Training** — standard scvi-tools training loop with multi-modal loss
4. **Per-modality diagnostics** — recon_loss, KL divergence, and Z variance per modality to assess training dynamics
5. **Three UMAP views** — joint Z, RNA-only Z, and ATAC-only Z reveal modality-specific and shared cell identity
6. **Decoder attribution analysis** — Jacobian-based attribution reveals which Z dims each decoder uses, enabling modality-specific views of cell heterogeneity

### Key design choices

- **GammaPoisson for ATAC** — count-based likelihood handles binary peak data naturally (0 and 1 are valid counts). More flexible than Bernoulli: can model count variation in fragment-level ATAC data.
- **Concatenation mode** — Z = [Z_rna; Z_atac] preserves modality-specific signal. Cells differing in chromatin but not transcriptome will separate in ATAC Z but not RNA Z.
- **Per-modality n_latent** — RNA typically gets more latent dimensions (higher feature complexity) while ATAC gets fewer (lower effective dimensionality).
- **No adversarial training** — simpler optimisation, relies on shared architecture and batch correction instead.
- **Attribution analysis** — post-training Jacobian attribution identifies modality contributions without modifying the model.

### Alternative Z strategies

```python
# Weighted mean (MultiVI-style) — single shared Z, n_latent must be equal
model = RegularizedMultimodalVI(mdata, latent_mode='weighted_mean', n_latent=64)

# Single encoder — one encoder for concatenated input
model = RegularizedMultimodalVI(mdata, latent_mode='single_encoder')
```